# Refinitiv Data Download — Full IDX Universe (Survivorship-Bias-Free)

In [1]:
import refinitiv.data as rd
import pandas as pd
from datetime import datetime, timedelta
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

rd.open_session()

<refinitiv.data.session.Definition object at 0x128199a60 {name='workspace'}>

## Configuration

## 1. Reconstruct Historical JCI Constituents (Survivorship-Bias-Free)

**Problem**: `0#.JKSE` returns only *current* index members. Any stock that was in the JCI
during 2008–2024 but later got delisted or removed is invisible — exactly the Harvey (1995)
backtracking bias.

**Solution**: Following LSEG's official methodology, we:
1. Take **yearly snapshots** of the index via `0#.JKSE(YYYYMMDD)` to see who was in the index at each year-end
2. Pull **joiner/leaver records** via `TR.IndexJL*` fields for exact entry/exit dates
3. **Union all RICs** ever observed into the download universe

In [3]:
# ── 1a. Yearly snapshots: who was in the JCI at each year-end? ──
# 0#.JKSE(YYYYMMDD) resolves the chain as it existed on that date.

snapshot_dates = [f"{y}1231" for y in range(start_date.year, end_date.year + 1)]
snapshots = {}       # {date_str: [ric, ...]}
all_rics_set = set()
failed_snapshots = []

for date_str in snapshot_dates:
    try:
        df = rd.get_data(
            universe=[f"0#.JKSE({date_str})"],
            fields=["TR.PriceClose"],
            parameters={"SDATE": date_str[:4]+"-12-31", "EDATE": date_str[:4]+"-12-31"}
        )
        if df is not None and not df.empty:
            rics = [r for r in df['Instrument'].unique() if r and str(r).strip()]
            snapshots[date_str] = rics
            all_rics_set.update(rics)
            print(f"  {date_str}: {len(rics)} constituents")
        else:
            print(f"  {date_str}: empty response")
            failed_snapshots.append(date_str)
    except Exception as e:
        failed_snapshots.append(date_str)
        print(f"  {date_str}: FAILED — {str(e)[:80]}")
    time.sleep(SLEEP_BETWEEN)

print(f"\nSnapshots retrieved: {len(snapshots)}/{len(snapshot_dates)}")
print(f"Unique RICs from snapshots: {len(all_rics_set)}")
if failed_snapshots:
    print(f"Failed snapshots: {failed_snapshots}")

  20081231: 912 constituents
  20091231: 912 constituents
  20101231: 912 constituents
  20111231: 912 constituents
  20121231: 912 constituents
  20131231: 912 constituents
  20141231: 912 constituents
  20151231: 912 constituents
  20161231: 912 constituents
  20171231: 912 constituents
  20181231: 912 constituents
  20191231: 912 constituents
  20201231: 912 constituents
  20211231: 912 constituents
  20221231: 912 constituents
  20231231: 912 constituents
  20241231: 912 constituents
  20251231: 912 constituents

Snapshots retrieved: 18/18
Unique RICs from snapshots: 912


In [ ]:
# ── 1b. Joiner/Leaver records ──
jl_frames = []
jl_starts = list(range(start_date.year, end_date.year, 5))

for yr in jl_starts:
    yr_end = min(yr + 4, end_date.year)
    try:
        df = rd.get_data(
            universe=[".JKSE"],
            fields=[
                "TR.IndexJLConstituentChangeDate",
                "TR.IndexJLConstituentRIC",
                "TR.IndexJLConstituentName",
            ],
            parameters={"SDATE": f"{yr}-01-01", "EDATE": f"{yr_end}-12-31", "IC": "B"}
        )
        if df is not None and not df.empty:
            jl_frames.append(df)
            print(f"  {yr}–{yr_end}: {len(df)} events")
    except Exception as e:
        print(f"  {yr}–{yr_end}: FAILED — {str(e)[:80]}")
    time.sleep(SLEEP_BETWEEN)

if jl_frames:
    jl_raw = pd.concat(jl_frames, ignore_index=True)
    jl_raw = jl_raw.rename(columns={'Date': 'Change_Date'})
    jl_raw['Change_Date'] = pd.to_datetime(jl_raw['Change_Date'], errors='coerce')
    jl_raw = jl_raw.dropna(subset=['Change_Date', 'RIC'])

    new_from_jl = set(jl_raw['RIC'].unique()) - all_rics_set
    all_rics_set.update(jl_raw['RIC'].unique())
    print(f"\nJL events: {len(jl_raw)} | New RICs from JL: {len(new_from_jl)}")
else:
    jl_raw = pd.DataFrame(columns=['Change_Date', 'RIC', 'Name'])
    print("No joiner/leaver data retrieved.")

In [7]:
if jl_frames:
    jl_raw = pd.concat(jl_frames, ignore_index=True)
    jl_raw = jl_raw.rename(columns={'Date': 'Change_Date', 'RIC': 'Constituent RIC'})
    jl_raw['Change_Date'] = pd.to_datetime(jl_raw['Change_Date'], errors='coerce')
    jl_raw = jl_raw.dropna(subset=['Change_Date', 'Constituent RIC'])

    new_from_jl = set(jl_raw['Constituent RIC'].unique()) - all_rics_set
    all_rics_set.update(jl_raw['Constituent RIC'].unique())
    print(f"\nJL events: {len(jl_raw)} | New RICs from JL: {len(new_from_jl)}")
else:
    jl_raw = pd.DataFrame(columns=['Change_Date', 'Constituent RIC', 'Name'])
    print("No joiner/leaver data retrieved.")


JL events: 738 | New RICs from JL: 80


In [8]:
# ── 1c. Assemble the full historical universe ──

# Also grab current constituents as a fallback / latest snapshot
try:
    current = rd.get_data(
        universe=["0#.JKSE"],
        fields=["TR.RIC", "TR.CommonName"]
    )
    current_rics = set(r for r in current["RIC"].unique() if r and str(r).strip())
    all_rics_set.update(current_rics)
    print(f"Current constituents added: {len(current_rics)}")
except Exception as e:
    print(f"Current constituent pull failed: {str(e)[:80]}")

all_rics = sorted(all_rics_set)

print(f"\n{'='*50}")
print(f"FULL HISTORICAL UNIVERSE: {len(all_rics)} unique RICs")
print(f"{'='*50}")
print(f"First 10: {all_rics[:10]}")
print(f"Last 10:  {all_rics[-10:]}")

Current constituents added: 912

FULL HISTORICAL UNIVERSE: 992 unique RICs
First 10: ['AADI.JK', 'AALI.JK', 'ABBA.JK', 'ABDA.JK', 'ABMM.JK', 'ACES.JK', 'ACRO.JK', 'ACST.JK', 'ADCP.JK', 'ADES.JK']
Last 10:  ['YELO.JK', 'YOII.JK', 'YPAS.JK', 'YULE.JK', 'YUPI.JK', 'ZATA.JK', 'ZBRA.JK', 'ZINC.JK', 'ZONE.JK', 'ZYRX.JK']


## 1d. Index Turnover Analysis
Quantifies how much the JCI membership changed year-over-year.
If turnover is low, survivorship bias is a minor concern.
If turnover is high, the bias from using only current constituents would be severe.

In [9]:
# ── Turnover from snapshot diffs ──
sorted_dates = sorted(snapshots.keys())
turnover_rows = []

for i in range(1, len(sorted_dates)):
    prev_date = sorted_dates[i - 1]
    curr_date = sorted_dates[i]
    prev_set = set(snapshots[prev_date])
    curr_set = set(snapshots[curr_date])

    joiners = curr_set - prev_set
    leavers = prev_set - curr_set
    avg_size = (len(prev_set) + len(curr_set)) / 2
    turnover_rate = (len(joiners) + len(leavers)) / avg_size * 100 if avg_size > 0 else 0

    turnover_rows.append({
        'Year': curr_date[:4],
        'Prev_Size': len(prev_set),
        'Curr_Size': len(curr_set),
        'Joiners': len(joiners),
        'Leavers': len(leavers),
        'Stable': len(prev_set & curr_set),
        'Turnover_Pct': round(turnover_rate, 1)
    })

turnover_df = pd.DataFrame(turnover_rows)

print("JCI CONSTITUENT TURNOVER (from yearly snapshots)")
print("=" * 70)
print(turnover_df.to_string(index=False))
print(f"\nAverage annual turnover: {turnover_df['Turnover_Pct'].mean():.1f}%")
print(f"Total unique RICs ever in index: {len(all_rics)}")
if snapshots:
    only_current = len(set(snapshots[sorted_dates[-1]]))
    missed = len(all_rics) - only_current
    print(f"Current-only approach would miss: ~{missed} stocks ({missed/len(all_rics)*100:.1f}% of universe)")

JCI CONSTITUENT TURNOVER (from yearly snapshots)
Year  Prev_Size  Curr_Size  Joiners  Leavers  Stable  Turnover_Pct
2009        912        912        0        0     912           0.0
2010        912        912        0        0     912           0.0
2011        912        912        0        0     912           0.0
2012        912        912        0        0     912           0.0
2013        912        912        0        0     912           0.0
2014        912        912        0        0     912           0.0
2015        912        912        0        0     912           0.0
2016        912        912        0        0     912           0.0
2017        912        912        0        0     912           0.0
2018        912        912        0        0     912           0.0
2019        912        912        0        0     912           0.0
2020        912        912        0        0     912           0.0
2021        912        912        0        0     912           0.0
2022        9

## 2. Daily Price & Volume Data

In [ ]:
start_date = datetime(2008, 1, 1)
end_date   = datetime(2025, 12, 31)

DAILY_BATCH   = 15   # RICs per get_history call (lower for local Workspace)
FUND_BATCH    = 30   # RICs per get_data call
SLEEP_BETWEEN = 3    # seconds between batches
YEARS_PER_CHUNK = 3  # years per daily-data chunk

print(f"Period: {start_date.date()} to {end_date.date()}")
print(f"Daily batch: {DAILY_BATCH} | Fund batch: {FUND_BATCH}")

# Build year-chunk boundaries
chunk_starts = []
cs = start_date
while cs < end_date:
    ce = min(cs + timedelta(days=YEARS_PER_CHUNK * 365), end_date)
    chunk_starts.append((cs, ce))
    cs = ce + timedelta(days=1)

print(f"Year chunks: {len(chunk_starts)}")
for s, e in chunk_starts:
    print(f"  {s.date()} → {e.date()}")

Year chunks: 6
  2008-01-01 → 2010-12-31
  2011-01-01 → 2013-12-31
  2014-01-01 → 2016-12-31
  2017-01-01 → 2020-01-01
  2020-01-02 → 2023-01-01
  2023-01-02 → 2025-12-31


In [ ]:
daily_frames = []
failed_daily = []

# total batches = number of chunks * number of batches per chunk
# number of batches per chunk = ceil(total RICs / DAILY_BATCH)
# daily_batches_per_chunk = (len(all_rics) + DAILY_BATCH - 1) // 

# Basically, calculate the number of stocks per batch, then how many batches we need for all RICs, and multiply by the number of date chunks to get total batches for progress tracking.
total_batches = len(chunk_starts) * ((len(all_rics) + DAILY_BATCH - 1) // DAILY_BATCH)
batch_num = 0

for chunk_start, chunk_end in chunk_starts:
    for i in range(0, len(all_rics), DAILY_BATCH):
        batch = all_rics[i:i + DAILY_BATCH]
        batch_num += 1
        
        try:
            df = rd.get_history(
                universe=batch,
                fields=["TR.PriceClose", "TR.Volume"],
                interval="1D",
                start=chunk_start.strftime('%Y-%m-%d'),
                end=chunk_end.strftime('%Y-%m-%d')
            )
            
            if df is not None and not df.empty:
                daily_frames.append(df)
            
            if batch_num % 10 == 0:
                print(f"  Batch {batch_num}/{total_batches} done — "
                      f"{chunk_start.date()} to {chunk_end.date()}, "
                      f"RICs {i+1}-{i+len(batch)}")
                
        except Exception as e:
            failed_daily.append({
                'batch_start_idx': i,
                'rics': batch,
                'period': f"{chunk_start.date()} to {chunk_end.date()}",
                'error': str(e)
            })
            print(f"  ⚠ Batch {batch_num} FAILED: {str(e)[:80]}")
        
        time.sleep(SLEEP_BETWEEN)

print(f"\nDaily download complete: {len(daily_frames)} successful batches, {len(failed_daily)} failures")

KeyboardInterrupt: 

In [ ]:
daily_raw = pd.concat(daily_frames, axis=0).dropna(how='all')
daily_raw = daily_raw[~daily_raw.index.duplicated(keep='first')]

# Wide → long
daily_long = daily_raw.stack(level=0).reset_index()
daily_long.columns = ['Date', 'Instrument', 'Price', 'Volume']
daily_long = daily_long.sort_values(['Instrument', 'Date']).reset_index(drop=True)
daily_long[['Price', 'Volume']] = daily_long[['Price', 'Volume']].apply(pd.to_numeric, errors='coerce')

print(f"Daily panel: {len(daily_long):,} rows, {daily_long['Instrument'].nunique()} stocks")
print(f"Range: {daily_long['Date'].min().date()} to {daily_long['Date'].max().date()}")
daily_long.head()

## 3. Monthly Fundamentals

In [ ]:
fund_frames = []
failed_fund = []

for i in range(0, len(all_rics), FUND_BATCH):
    batch = all_rics[i:i + FUND_BATCH]
    batch_idx = i // FUND_BATCH + 1
    total = (len(all_rics) + FUND_BATCH - 1) // FUND_BATCH
    
    try:
        df = rd.get_data(
            universe=batch,
            fields=[
                "TR.CompanyMarketCapitalization.date",
                "TR.CompanyMarketCapitalization",
                "TR.F.ReturnAvgTotAssetsPctTTM.date",
                "TR.F.ReturnAvgTotAssetsPctTTM",
                "TR.IPODate",
                "TR.DPSMean.date",
                "TR.DPSMean",
                "TR.F.BookValuePerShr.date",
                "TR.F.BookValuePerShr",
                "TR.TRBCEconomicSector",
                "TR.SharesOutstanding.date",
                "TR.SharesOutstanding"
            ],
            parameters={
                'SDate': start_date.strftime('%Y-%m-%d'),
                'EDate': end_date.strftime('%Y-%m-%d'),
                'Frq': 'M'
            }
        )
        
        if df is not None and not df.empty:
            fund_frames.append(df)
        
        if batch_idx % 5 == 0 or batch_idx == total:
            print(f"  Fundamentals batch {batch_idx}/{total} done — RICs {i+1}-{i+len(batch)}")
            
    except Exception as e:
        failed_fund.append({
            'batch_start_idx': i,
            'rics': batch,
            'error': str(e)
        })
        print(f"  ⚠ Batch {batch_idx} FAILED: {str(e)[:80]}")
    
    time.sleep(SLEEP_BETWEEN)

print(f"\nFundamentals download complete: {len(fund_frames)} successful batches, {len(failed_fund)} failures")

In [ ]:
fund_raw = pd.concat(fund_frames, ignore_index=True)

# Rename positionally — Refinitiv returns duplicate 'Date' cols per time-varying field
fund_raw.columns = [
    'Instrument', 'Date', 'Market_Cap', 'Date_ROA', 'ROA',
    'IPO_Date', 'Date_DPS', 'DPS_Raw', 'Date_BVPS', 'BVPS',
    'Sector', 'Date_Shares', 'Shares_Outstanding'
]
fund_clean = fund_raw.drop(columns=['Date_ROA', 'Date_DPS', 'Date_BVPS', 'Date_Shares'])

# Type conversions
for col in ['Market_Cap', 'ROA', 'DPS_Raw', 'BVPS', 'Shares_Outstanding']:
    fund_clean[col] = pd.to_numeric(fund_clean[col], errors='coerce')
fund_clean['Date'] = pd.to_datetime(fund_clean['Date'], errors='coerce')
fund_clean['IPO_Date'] = pd.to_datetime(fund_clean['IPO_Date'], errors='coerce')

# Forward/back-fill static fields within each instrument
fund_clean['Sector'] = fund_clean['Sector'].replace(r'^\s*$', pd.NA, regex=True)
for col in ['IPO_Date', 'Sector']:
    fund_clean[col] = fund_clean.groupby('Instrument')[col].ffill().bfill()

# Derived variables
fund_clean['Market_Cap_Bil'] = fund_clean['Market_Cap'] / 1e9
fund_clean['Div_Payer'] = (fund_clean['DPS_Raw'].gt(0)).astype(int)
fund_clean['Months_Since_Listing'] = (
    (fund_clean['Date'] - fund_clean['IPO_Date']).dt.days / 30.44
).round().astype('Int64')

fund_clean = fund_clean.dropna(subset=['Date'])

print(f"Fundamentals: {len(fund_clean):,} rows, {fund_clean['Instrument'].nunique()} stocks")
print(f"\nCoverage (%):")
print(fund_clean.notnull().mean() * 100)

## CHECKPOINT — Save before memory runs out

In [ ]:
# Save everything we have so far
daily_long.to_csv('idx_daily_prices.csv', index=False)
fund_clean.to_csv('idx_fundamentals.csv', index=False)
turnover_df.to_csv('idx_turnover_analysis.csv', index=False)
pd.DataFrame({'RIC': all_rics}).to_csv('idx_ric_list.csv', index=False)

if not jl_raw.empty:
    jl_raw.to_csv('idx_joiner_leaver_events.csv', index=False)

snapshot_rows = []
for date_str, rics in snapshots.items():
    for ric in rics:
        snapshot_rows.append({'Snapshot_Date': date_str, 'RIC': ric})
pd.DataFrame(snapshot_rows).to_csv('idx_constituent_snapshots.csv', index=False)

print("Checkpoint saved:")
print(f"  idx_daily_prices.csv        — {len(daily_long):,} rows")
print(f"  idx_fundamentals.csv        — {len(fund_clean):,} rows")
print(f"  idx_turnover_analysis.csv")
print(f"  idx_ric_list.csv            — {len(all_rics)} RICs")
print(f"  idx_joiner_leaver_events.csv")
print(f"  idx_constituent_snapshots.csv")

## 4. Download JCI Index Returns

In [ ]:
# .JKSE is the Refinitiv RIC for the Jakarta Composite Index
jci_raw = rd.get_history(
    universe=[".JKSE"],
    fields=["TR.PriceClose"],
    interval="1D",
    start=start_date.strftime('%Y-%m-%d'),
    end=end_date.strftime('%Y-%m-%d')
)

jci = jci_raw.reset_index()
jci.columns = ['Date', 'JCI_Close']
jci['JCI_Close'] = pd.to_numeric(jci['JCI_Close'], errors='coerce')
jci['Market_Return'] = np.log(jci['JCI_Close'] / jci['JCI_Close'].shift(1))
jci = jci.dropna(subset=['Market_Return']).reset_index(drop=True)

print(f"JCI index: {len(jci):,} trading days")
print(f"Date range: {jci['Date'].min().date()} to {jci['Date'].max().date()}")
jci.head()

## 5. Download Bank Indonesia Risk-Free Rate\nBI 7-Day Reverse Repo Rate (now called BI-Rate), expanded to a daily series via forward-fill.

In [ ]:
# IDCBRR=ECI is the Refinitiv economic indicator for the BI repo rate
rf_raw = rd.get_history(
    universe=["IDCBRR=ECI"],
    fields=["VALUE"],
    interval="monthly",
    start=start_date.strftime('%Y-%m-%d'),
    end=end_date.strftime('%Y-%m-%d')
)

# Expand monthly rate to daily via forward-fill
daily_range = pd.date_range(start=start_date, end=end_date, freq='D')
rf_daily = rf_raw.reindex(daily_range).ffill().bfill().reset_index()
rf_daily.columns = ['Date', 'Annual_BI_Rate']
rf_daily['Annual_BI_Rate'] = pd.to_numeric(rf_daily['Annual_BI_Rate'], errors='coerce')

# Convert annualized % to daily decimal (IDR money market convention: /360)
rf_daily['Daily_Rf'] = (rf_daily['Annual_BI_Rate'] / 100) / 360

print(f"Risk-free rate: {len(rf_daily):,} daily observations")
print(f"BI Rate range: {rf_daily['Annual_BI_Rate'].min()}% – {rf_daily['Annual_BI_Rate'].max()}%")
rf_daily.head()

## 6. Assemble Master Panel\nMerge daily prices with JCI returns and risk-free rate. Calculate log returns and excess returns.

In [ ]:
# Ensure Date columns are datetime
daily_long['Date'] = pd.to_datetime(daily_long['Date'])
jci['Date'] = pd.to_datetime(jci['Date'])
rf_daily['Date'] = pd.to_datetime(rf_daily['Date'])

# Merge daily stock data with JCI market returns
master = pd.merge(daily_long, jci[['Date', 'Market_Return']], on='Date', how='left')

# Merge with risk-free rate
master = pd.merge(master, rf_daily[['Date', 'Daily_Rf']], on='Date', how='left')

# Calculate individual stock log returns
master = master.sort_values(['Instrument', 'Date'])
master['Stock_Return'] = master.groupby('Instrument')['Price'].transform(
    lambda x: np.log(x / x.shift(1))
)

# Excess returns
master['Excess_Return'] = master['Stock_Return'] - master['Daily_Rf']
master['Market_Excess'] = master['Market_Return'] - master['Daily_Rf']

# Day of week (0=Monday, 4=Friday)
master['DayOfWeek'] = master['Date'].dt.dayofweek
master['DayName'] = master['Date'].dt.day_name()

# Year-Month for merging with monthly fundamentals
master['YearMonth'] = master['Date'].dt.to_period('M')

print(f"Master panel: {len(master):,} rows, {master['Instrument'].nunique()} stocks")
print(f"\nColumn list: {list(master.columns)}")
print(f"\nSample coverage:")
print(master[['Price', 'Volume', 'Stock_Return', 'Market_Return', 'Daily_Rf']].notnull().mean() * 100)
master.head(10)

## 7. Retry Failed Batches

In [ ]:
def retry_batch(fail_list, fetch_fn, label=""):
    """Retry a list of failed batches using fetch_fn(fail_entry) -> DataFrame or None."""
    if not fail_list:
        print(f"No failed {label} batches to retry.")
        return []
    print(f"Retrying {len(fail_list)} failed {label} batches...")
    recovered, still_failed = [], []
    for fail in fail_list:
        try:
            df = fetch_fn(fail)
            if df is not None and not df.empty:
                recovered.append(df)
                print(f"  ✓ Recovered batch idx {fail['batch_start_idx']}")
            else:
                still_failed.append(fail)
        except Exception as e:
            still_failed.append(fail)
            print(f"  ✗ Still failed: {str(e)[:60]}")
        time.sleep(SLEEP_BETWEEN * 2)
    if still_failed:
        print(f"  {len(still_failed)} permanently failed")
    return recovered

# Retry daily
def _fetch_daily(fail):
    s, e = fail['period'].split(' to ')
    return rd.get_history(universe=fail['rics'], fields=["TR.PriceClose", "TR.Volume"],
                          interval="1D", start=s, end=e)

recovered_daily = retry_batch(failed_daily, _fetch_daily, "daily")
if recovered_daily:
    retry_raw = pd.concat(recovered_daily, axis=0).dropna(how='all')
    retry_long = retry_raw.stack(level=0).reset_index()
    retry_long.columns = ['Date', 'Instrument', 'Price', 'Volume']
    retry_long[['Price', 'Volume']] = retry_long[['Price', 'Volume']].apply(pd.to_numeric, errors='coerce')
    daily_long = (pd.concat([daily_long, retry_long], ignore_index=True)
                  .drop_duplicates(subset=['Date', 'Instrument'], keep='first')
                  .sort_values(['Instrument', 'Date']).reset_index(drop=True))
    print(f"  +{len(retry_long):,} rows recovered. New total: {len(daily_long):,}")

# Retry fundamentals
FUND_FIELDS = [
    "TR.CompanyMarketCapitalization.date", "TR.CompanyMarketCapitalization",
    "TR.F.ReturnAvgTotAssetsPctTTM.date", "TR.F.ReturnAvgTotAssetsPctTTM",
    "TR.IPODate", "TR.DPSMean.date", "TR.DPSMean",
    "TR.F.BookValuePerShr.date", "TR.F.BookValuePerShr",
    "TR.TRBCEconomicSector", "TR.SharesOutstanding.date", "TR.SharesOutstanding"
]

def _fetch_fund(fail):
    return rd.get_data(universe=fail['rics'], fields=FUND_FIELDS,
                       parameters={'SDate': start_date.strftime('%Y-%m-%d'),
                                   'EDate': end_date.strftime('%Y-%m-%d'), 'Frq': 'M'})

recovered_fund = retry_batch(failed_fund, _fetch_fund, "fundamentals")
if recovered_fund:
    print("  Re-run Section 3 cleaning cell to incorporate.")

## 8. Save to CSV

In [ ]:
# Save all datasets
daily_long.to_csv('idx_daily_prices.csv', index=False)
fund_clean.to_csv('idx_fundamentals.csv', index=False)
jci.to_csv('jci_index.csv', index=False)
rf_daily.to_csv('bi_riskfree_rate.csv', index=False)

# Save master panel (daily returns merged with market and rf)
master.to_csv('idx_master_panel.csv', index=False)

# Save the full historical RIC list
pd.DataFrame({'RIC': all_rics}).to_csv('idx_ric_list.csv', index=False)

# Save turnover analysis
turnover_df.to_csv('idx_turnover_analysis.csv', index=False)

# Save joiner/leaver records
if not jl_raw.empty:
    jl_raw.to_csv('idx_joiner_leaver_events.csv', index=False)

# Save snapshot membership for reproducibility
snapshot_rows = []
for date_str, rics in snapshots.items():
    for ric in rics:
        snapshot_rows.append({'Snapshot_Date': date_str, 'RIC': ric})
if snapshot_rows:
    pd.DataFrame(snapshot_rows).to_csv('idx_constituent_snapshots.csv', index=False)

print("Saved files:")
for f in ['idx_daily_prices.csv', 'idx_fundamentals.csv', 'jci_index.csv',
          'bi_riskfree_rate.csv', 'idx_master_panel.csv', 'idx_ric_list.csv',
          'idx_turnover_analysis.csv', 'idx_joiner_leaver_events.csv',
          'idx_constituent_snapshots.csv']:
    print(f"  {f}")

## 9. Data Quality Report

In [ ]:
print("=" * 60)
print("DATA QUALITY REPORT")
print("=" * 60)

# Survivorship bias
print(f"\n── Survivorship Bias ──")
print(f"Historical universe:   {len(all_rics)} unique RICs")
if snapshots:
    latest_snap = snapshots[sorted(snapshots.keys())[-1]]
    missed = len(all_rics) - len(latest_snap)
    print(f"Current constituents:  {len(latest_snap)}")
    print(f"Delisted/removed:      ~{missed} ({missed/len(all_rics)*100:.1f}%)")
    print(f"Avg annual turnover:   {turnover_df['Turnover_Pct'].mean():.1f}%")
if not jl_raw.empty:
    print(f"JL change events:      {len(jl_raw)}")

# Daily prices
print(f"\n── Daily Prices ──")
print(f"Rows: {len(daily_long):,} | Stocks: {daily_long['Instrument'].nunique()}")
print(f"Range: {daily_long['Date'].min().date()} to {daily_long['Date'].max().date()}")
obs = daily_long.groupby('Instrument').size()
print(f"Obs/stock: median={obs.median():.0f}, min={obs.min()}, max={obs.max()}")

# Fundamentals
print(f"\n── Fundamentals ──")
print(f"Rows: {len(fund_clean):,} | Stocks: {fund_clean['Instrument'].nunique()}")
print(f"Sectors: {fund_clean['Sector'].nunique()} | Div payers: {fund_clean.groupby('Instrument')['Div_Payer'].max().sum()}")

# JCI & Rf
print(f"\n── Market Data ──")
print(f"JCI trading days: {len(jci):,}")
print(f"BI Rate range: {rf_daily['Annual_BI_Rate'].min():.2f}% – {rf_daily['Annual_BI_Rate'].max():.2f}%")

# Failures
n_fail = len(failed_snapshots) + len(failed_daily) + len(failed_fund)
if n_fail > 0:
    print(f"\n── Failures ──")
    print(f"Snapshots: {len(failed_snapshots)} | Daily: {len(failed_daily)} | Fund: {len(failed_fund)}")
else:
    print(f"\nNo download failures.")